# 01 · Base de conhecimento e casos de teste

Guiding Hand · Generative AI for Engineering

Este notebook carrega a base (`base/trechos.jsonl`), resume fontes e licenças e confere o gabarito dos casos de teste (`avaliacao/casos_teste.jsonl`). Rodar com `Kernel > Restart & Run All` para regenerar as saídas.

In [1]:
import sys, json, unicodedata
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from config import TRECHOS, CASOS

trechos = pd.read_json(TRECHOS, lines=True)
casos = pd.read_json(CASOS, lines=True)
print(f'{len(trechos)} trechos | {len(casos)} casos de teste')

32 trechos | 35 casos de teste


## 1. Tabela da base (fonte, formato, licença, nº de trechos)

In [2]:
tabela = (trechos.groupby('norma')
          .agg(trechos=('id', 'count'), licenca=('licenca', 'first'))
          .sort_values('trechos', ascending=False))
tabela['formato'] = 'Texto de lei → JSONL'
print(tabela[['formato', 'licenca', 'trechos']].to_string())
print('\nTotal de trechos:', tabela.trechos.sum())

                              formato                                        licenca  trechos
norma                                                                                        
Lei 13.146/2015  Texto de lei → JSONL  Domínio público (Lei 9.610/1998, art. 8º, IV)       24
Lei 10.098/2000  Texto de lei → JSONL  Domínio público (Lei 9.610/1998, art. 8º, IV)        4
Lei 10.048/2000  Texto de lei → JSONL  Domínio público (Lei 9.610/1998, art. 8º, IV)        1
Lei 10.257/2001  Texto de lei → JSONL  Domínio público (Lei 9.610/1998, art. 8º, IV)        1
Lei 11.126/2005  Texto de lei → JSONL  Domínio público (Lei 9.610/1998, art. 8º, IV)        1
Lei 12.587/2012  Texto de lei → JSONL  Domínio público (Lei 9.610/1998, art. 8º, IV)        1

Total de trechos: 32


## 2. Tamanho dos trechos
Trechos curtos (um artigo ou parte dele) deixam a busca mais precisa e a citação mais fácil de conferir.

In [3]:
trechos['palavras'] = trechos.texto.str.split().str.len()
print(trechos.palavras.describe().round(1).to_string())
print('\nTrechos por tema:')
print(trechos.tema.value_counts().to_string())

count     32.0
mean      51.1
std       25.5
min       18.0
25%       32.8
50%       42.5
75%       69.0
max      111.0

Trechos por tema:
tema
definição                  4
atendimento prioritário    3
transporte                 3
táxi                       2
calçadas                   2
obstáculos                 2
discriminação              1
denúncia                   1
benefício                  1
ingresso                   1
estacionamento             1
locadoras                  1
acessibilidade             1
desenho universal          1
obras na via               1
internet                   1
tecnologia assistiva       1
crime                      1
dados                      1
atendimento                1
cão-guia                   1
semáforo                   1


## 3. Exemplos de trechos

In [4]:
for _, t in trechos.sample(3, random_state=42).iterrows():
    print(f"[{t.id}] {t.fonte}")
    print(t.texto[:300] + ('...' if len(t.texto) > 300 else ''))
    print()

[T30] Lei 10.098/2000, art. 2º, VII
mobiliário urbano: conjunto de objetos existentes nas vias e nos espaços públicos, superpostos ou adicionados aos elementos de urbanização ou de edificação, de forma que sua modificação ou seu traslado não provoque alterações substanciais nesses elementos, tais como semáforos, postes de sinalização ...

[T16] Lei 13.146/2015, art. 52, caput
As locadoras de veículos são obrigadas a oferecer 1 (um) veículo adaptado para uso de pessoa com deficiência, a cada conjunto de 20 (vinte) veículos de sua frota.

[T25] Lei 11.126/2005, art. 1º, caput e § 2º
É assegurado à pessoa com deficiência visual acompanhada de cão-guia o direito de ingressar e de permanecer com o animal em todos os meios de transporte e em estabelecimentos abertos ao público, de uso público e privados de uso coletivo, desde que observadas as condições impostas por esta Lei. O dis...



## 4. Casos de teste com gabarito
- `resposta`: a base tem a resposta; o gabarito indica o trecho esperado e palavras-chave.
- `nao_consta`: a base **não** tem a resposta; o assistente deve recusar. Há perguntas quase-vizinhas de propósito, para medir alucinação.

In [5]:
print(casos.tipo.value_counts().to_string())
print()
print(casos[casos.tipo == 'nao_consta'][['id', 'pergunta', 'motivo']].to_string(index=False))

tipo
resposta      27
nao_consta     8

 id                                                                          pergunta                                                      motivo
N01                                            Qual o horário do ônibus 875P-10 hoje?                            dado em tempo real, fora da base
N02 Quanto custa a passagem de metrô para pessoa com deficiência visual em São Paulo?                              tarifa municipal, fora da base
N03                  Qual o valor em reais da multa por estacionar em vaga reservada? quase-vizinha: a base fala das vagas, não do valor da multa
N04                                        Como solicito o passe livre interestadual?                                     outra lei, fora da base
N05                                   Posso levar meu cachorro de estimação no metrô?                  quase-vizinha: a base só trata de cão-guia
N06                    Qual o prazo para a prefeitura consertar uma calçada quebrada

## 5. Conferência automática do gabarito
Toda palavra-chave precisa existir no trecho indicado. Se algum caso falhar, o gabarito está errado, e não o modelo.

In [6]:
def norm(s): return unicodedata.normalize('NFC', s).lower()
mapa = dict(zip(trechos.id, trechos.texto))
falhas = []
for _, c in casos[casos.tipo == 'resposta'].iterrows():
    for kw in c.palavras_chave:
        if norm(kw) not in norm(mapa[c.trecho_esperado]):
            falhas.append((c.id, kw))
print('Falhas de gabarito:', falhas if falhas else 'nenhuma')
print('Trechos citados pelo gabarito:', casos.trecho_esperado.dropna().nunique(), 'de', len(trechos))

Falhas de gabarito: nenhuma
Trechos citados pelo gabarito: 26 de 32
